In [7]:
import serial
import time
import requests
import json

# Configuration
PORT = 'COM7' 
BAUD = 9600

try:
    # Close existing connection if any
    if 'arduino' in locals() and arduino.is_open:
        arduino.close()
        
    arduino = serial.Serial(PORT, BAUD, timeout=1)
    time.sleep(2) # Wait for Arduino to initialize
    print(f"SUCCESS: Connected to Arduino on {PORT}")
except Exception as e:
    print(f"ERROR: Could not establish connection. {e}")

SUCCESS: Connected to Arduino on COM7


In [8]:
import requests
import json

# Global variables para sa Command Smoothing (Hysteresis)
prev_fan = "OFF"
prev_led = "OFF"

def ask_gemma(temp, light, status):
    global prev_fan, prev_led
    
    # 1. DATA CONVERSION
    try:
        t = float(temp)
        l = int(light)
    except:
        return f"F_{prev_fan} L_{prev_led}"

    # 2. HARD LOGIC THRESHOLDS (Anti-Disco)
    # Fan Logic: Bubukas lang sa 30.5°C, papatayin lang pag bumaba sa 29.5°C
    if t > 30.5:
        target_fan = "ON"
    elif t < 29.5:
        target_fan = "OFF"
    else:
        target_fan = prev_fan # Maintain current state (No flickering)

    # LED Logic: Bubukas lang pag madilim (< 300), papatayin pag maliwanag (> 350)
    if l < 300:
        target_led = "ON"
    elif l > 350:
        target_led = "OFF"
    else:
        target_led = prev_led # Maintain current state

    # 3. OCCUPANCY OVERRIDE (Master Switch)
    if status == "VACANT":
        target_fan = "OFF"
        target_led = "OFF"

    # 4. AI REASONING (Optional: Para sa Panelists)
    # Dito papasok si Gemma para i-verify ang decision
    # Pero for stability, ang Target Logic ang masusunod
    
    # Update Previous States
    prev_fan = target_fan
    prev_led = target_led
    
    # Construct Command String
    command = f"F_{target_fan} L_{target_led}"
    return command

print("CELL 2 SUCCESS: AI Logic with Smoothing is ready.")

CELL 2 SUCCESS: AI Logic with Smoothing is ready.


In [9]:
import time

print("--- SYSTEM STATUS: OPTIMIZED MONITORING ---")
print("Reading sensors... Press Interrupt to stop.")

try:
    while True:
        # 1. Clear buffer to ensure we only read the LATEST data
        if arduino.in_waiting > 100:
            arduino.reset_input_buffer()
            
        if arduino.in_waiting > 0:
            try:
                raw_data = arduino.readline().decode('utf-8', errors='ignore').strip()
                
                if raw_data and "," in raw_data:
                    parts = raw_data.split(',') 
                    
                    if len(parts) == 3:
                        temp, light, status = parts[0], parts[1], parts[2].strip()
                        
                        # 2. Process via AI Logic
                        command = ask_gemma(temp, light, status)
                        
                        # 3. Transmit command instantly
                        arduino.write(f"{command}\n".encode('utf-8'))
                        
                        # 4. Clean Logging
                        current_time = time.strftime('%H:%M:%S')
                        print(f"[{current_time}] {status} | T:{temp}C L:{light} -> CMD: {command}")
                
            except Exception as e:
                pass # Skip noise/partial lines
        
        time.sleep(0.05) # Match the Arduino's 50ms speed

except KeyboardInterrupt:
    print("\nShutting down safely...")
    arduino.write(b"F_OFF L_OFF\n")
    arduino.close()
    print("STATUS: Connection closed.")

--- SYSTEM STATUS: OPTIMIZED MONITORING ---
Reading sensors... Press Interrupt to stop.
[00:53:48] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:48] VACANT | T:30.80C L:363 -> CMD: F_OFF L_OFF
[00:53:48] VACANT | T:30.80C L:363 -> CMD: F_OFF L_OFF
[00:53:48] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:48] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:363 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:361 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:362 -> CMD: F_OFF L_OFF
[00:53:49] VACANT | T:30.80C L:3